%md
## Read: scanned PDFs (OCR)
Runs ai_parse_document on PDFs routed to `ocr` in pdf_profile, saves raw output
to `ocr_raw`, then parses each HTML table into `pdf_rows` (one row per table row,
cells as an array). Header rows are skipped; names come from ECI.

**2026 result:** 9 files, every page returned a table. 2,410 of 2,799 rows at the
expected width. 389 off-width rows (padded with blanks, a duplicated cell, or a
dropped zero) are kept as-is here and repaired in silver using row and column tallies.

In [0]:
from pyspark.sql.functions import expr, current_timestamp

ocr_targets = (
    spark.table("tnelections.form20_bronze.pdf_profile")
    .filter("read_method = 'ocr'")
    .select("year", "ac_no")
)

ocr_raw = (
    spark.table("tnelections.form20_bronze.pdf_files")
    .join(ocr_targets, ["year", "ac_no"])
    .withColumn("parsed", expr("ai_parse_document(content, map('version', '2.0'))"))
    .select("year", "ac_no", "parsed")
    .withColumn("parsed_at", current_timestamp())
)

(ocr_raw.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("tnelections.form20_bronze.ocr_raw"))

In [0]:
display(spark.sql("""
    WITH ocr_pages AS (
        SELECT r.year, r.ac_no,
               COUNT(DISTINCT el.value:bbox[0]:page_id::int)  AS pages_read,
               COUNT_IF(el.value:type::string = 'table')      AS tables
        FROM tnelections.form20_bronze.ocr_raw r,
             LATERAL variant_explode(r.parsed:document:elements) AS el
        GROUP BY r.year, r.ac_no
    )
    SELECT p.ac_no, p.page_count, o.pages_read, o.tables
    FROM tnelections.form20_bronze.pdf_profile p
    JOIN ocr_pages o USING (year, ac_no)
    ORDER BY p.ac_no
"""))

In [0]:
from html.parser import HTMLParser

class TableRows(HTMLParser):
    def __init__(self):
        super().__init__()
        self.rows, self.row, self.cell, self.in_td = [], None, "", False

    def handle_starttag(self, tag, attrs):
        if tag == "tr":
            self.row = []
        elif tag == "td" and self.row is not None:
            self.in_td, self.cell = True, ""
        elif tag == "br" and self.in_td:
            self.cell += " "

    def handle_endtag(self, tag):
        if tag == "td" and self.in_td:
            self.row.append(self.cell.strip())
            self.in_td = False
        elif tag == "tr" and self.row is not None:
            if self.row:                  # header rows have no <td>, so they're skipped
                self.rows.append(self.row)
            self.row = None

    def handle_data(self, data):
        if self.in_td:
            self.cell += data

def table_rows(html):
    if not html:
        return []
    parser = TableRows()
    parser.feed(html)
    return parser.rows

In [0]:
test = spark.sql("""
    SELECT el.value:content::string AS html
    FROM tnelections.form20_bronze.ocr_raw r,
         LATERAL variant_explode(r.parsed:document:elements) AS el
    WHERE r.year = 2026 AND r.ac_no = 154
      AND el.value:type::string = 'table'
      AND el.value:bbox[0]:page_id::int = 20
""").first()

for r in table_rows(test.html):
    print(len(r), r)

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS tnelections.form20_bronze.pdf_rows (
        year INT, ac_no INT, page INT, table_idx INT, row_idx INT,
        cells ARRAY<STRING>, n_cells INT, read_method STRING, read_at TIMESTAMP
    )
    COMMENT 'One row per table row from either reader (ocr or text), cells kept as text'
""")

In [0]:
from pyspark.sql.functions import udf, col, posexplode, size, lit, current_timestamp
from pyspark.sql.types import ArrayType, StringType

table_rows_udf = udf(table_rows, ArrayType(ArrayType(StringType())))

ocr_tables = spark.sql("""
    SELECT r.year, r.ac_no,
           el.value:bbox[0]:page_id::int AS page,
           el.pos                        AS table_idx,
           el.value:content::string      AS html
    FROM tnelections.form20_bronze.ocr_raw r,
         LATERAL variant_explode(r.parsed:document:elements) AS el
    WHERE el.value:type::string = 'table'
""")

ocr_rows = (
    ocr_tables
    .withColumn("rows", table_rows_udf(col("html")))
    .select("year", "ac_no", "page", "table_idx",
            posexplode("rows").alias("row_idx", "cells"))
    .withColumn("n_cells", size("cells"))
    .withColumn("read_method", lit("ocr"))
    .withColumn("read_at", current_timestamp())
)

(ocr_rows.write
    .mode("overwrite")
    .option("replaceWhere", "read_method = 'ocr'")
    .saveAsTable("tnelections.form20_bronze.pdf_rows"))

In [0]:
display(spark.sql("""
    SELECT ac_no,
           COUNT(*)                AS rows,
           COUNT(DISTINCT n_cells) AS distinct_widths,
           MIN(n_cells)            AS min_cells,
           MAX(n_cells)            AS max_cells
    FROM tnelections.form20_bronze.pdf_rows
    WHERE read_method = 'ocr'
    GROUP BY ac_no
    ORDER BY ac_no
"""))

In [0]:
display(spark.sql("""
    SELECT ac_no, n_cells,
           COUNT(*)          AS rows,
           MIN(page)         AS first_page,
           MAX(page)         AS last_page,
           any_value(cells)  AS sample_row
    FROM tnelections.form20_bronze.pdf_rows
    WHERE read_method = 'ocr' AND ac_no IN (151, 155)
    GROUP BY ac_no, n_cells
    ORDER BY ac_no, rows DESC
"""))

In [0]:
display(spark.sql("""
    SELECT r.ac_no,
           SUM(size(regexp_extract_all(el.value:content::string, '<td[^>]*colspan', 0))) AS td_colspans,
           SUM(size(regexp_extract_all(el.value:content::string, '<td[^>]*rowspan', 0))) AS td_rowspans
    FROM tnelections.form20_bronze.ocr_raw r,
         LATERAL variant_explode(r.parsed:document:elements) AS el
    WHERE el.value:type::string = 'table'
    GROUP BY r.ac_no
    ORDER BY r.ac_no
"""))

In [0]:
display(spark.sql("""
    WITH widths AS (
        SELECT ac_no, n_cells, COUNT(*) AS cnt
        FROM tnelections.form20_bronze.pdf_rows
        WHERE read_method = 'ocr'
        GROUP BY ac_no, n_cells
    ),
    expected AS (
        SELECT ac_no, max_by(n_cells, cnt) AS expected_width
        FROM widths GROUP BY ac_no
    )
    SELECT r.ac_no, e.expected_width,
           COUNT_IF(r.n_cells = e.expected_width)      AS ok_rows,
           COUNT_IF(r.n_cells = e.expected_width + 1)  AS one_extra,
           COUNT_IF(r.n_cells = e.expected_width - 1)  AS one_missing,
           COUNT_IF(r.n_cells > e.expected_width + 1)  AS padded,
           COUNT_IF(r.n_cells < e.expected_width - 1)  AS very_short
    FROM tnelections.form20_bronze.pdf_rows r
    JOIN expected e USING (ac_no)
    WHERE r.read_method = 'ocr'
    GROUP BY r.ac_no, e.expected_width
    ORDER BY r.ac_no
"""))